In [ ]:
# 1. Single Technique Failure Demonstration (Otsu fails to isolate the cup alone)
single_technique_failure = otsu_seg.copy()

# 2. Composite Pipeline Step 1: Targeted Region Growing
cup_seed = (200, 260)
raw_cup_mask = region_growing(gray, cup_seed, threshold=18)

# 2. Composite Pipeline Step 2: Morphological Refinement
kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9))
closed_cup_mask = cv2.morphologyEx(raw_cup_mask, cv2.MORPH_CLOSE, kernel)
final_cup_mask = cv2.morphologyEx(closed_cup_mask, cv2.MORPH_OPEN, kernel)

# 3. Object Retention: Retain only the cup, set everything else to white
retained_cup = img_rgb.copy()
retained_cup[final_cup_mask == 0] = [255, 255, 255]

# 4. Object Disappearance: Inpaint cup region to make it disappear
disappeared_bgr = cv2.inpaint(img_bgr, final_cup_mask, inpaintRadius=9, flags=cv2.INPAINT_TELEA)
disappeared_rgb = cv2.cvtColor(disappeared_bgr, cv2.COLOR_BGR2RGB)

# 5. Visual Display
fig, axes = plt.subplots(2, 3, figsize=(15, 9))

axes[0, 0].imshow(img_rgb)
axes[0, 0].set_title("1. Original Image")

axes[0, 1].imshow(single_technique_failure, cmap='gray')
axes[0, 1].set_title("Single Method: Otsu (Fails to Isolate Cup)")

axes[0, 2].imshow(final_cup_mask, cmap='gray')
axes[0, 2].set_title("2. Proposed Pipeline Mask (Cup)")

axes[1, 0].imshow(retained_cup)
axes[1, 0].set_title("3. Object Retained (Cup Only)")

axes[1, 1].imshow(disappeared_rgb)
axes[1, 1].set_title("4. Object Disappeared (Inpainted Scene)")

# Boundary Overlay
overlay = img_rgb.copy()
overlay[final_cup_mask > 0] = [255, 40, 40]
axes[1, 2].imshow(cv2.addWeighted(img_rgb, 0.7, overlay, 0.3, 0))
axes[1, 2].set_title("5. Cup Boundary Overlay")

for ax in axes.ravel():
    ax.axis('off')
plt.tight_layout()
plt.show()
